# [14.1] JEPA and World-Model Controls - Solutions

**Core question.** When does a frozen video latent deserve a world-state interpretation?

This solved notebook follows the same ladder as the exercise notebook: toy
ground truth first, then bounded interpretation of the V-JEPA 2 generated-video
report.

## Learning Objectives

- Verify paired cosine and target-prediction helpers.
- Interpret collapse, shuffled-label, copy, shuffled-action, absent-object, and
  random-token controls.
- Read the CUDA signature result without overclaiming beyond generated videos.

> ```yaml
> Difficulty: 4
> Importance: 4
> ```

<details>
<summary>Help - how to read the controls</summary>

A pass only means something if the matched control fails. In this section the
controls are collapse, shuffled labels, copy rollout, shuffled actions, absent
objects, different objects, and random-token patches.

</details>

In [1]:
GT_TIER = "GT-1"
EXERCISE_ID = "14_1_jepa_and_world_model_controls"
DIFFICULTY = 4
IMPORTANCE = 4
EXPECTED_RUNTIME = "seconds for toy contracts; minutes for the CUDA V-JEPA 2 latent-control preflight"
REQUIRES_GPU = True

import json
import sys
from pathlib import Path

import torch as t
import torch.nn.functional as F

chapter = "chapter14_jepa_world_models"
section = "part1_jepa_world_model_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_jepa_world_model_controls.tests as tests
import part1_jepa_world_model_controls.utils as utils

from arena_ext.jepa_world_models import (
    collapse_diagnostics_report,
    jepa_prediction_report,
    latent_rollout_report,
    object_permanence_report,
    transition_consistency_report,
    world_state_probe_report,
)

MAIN = True

from chapter14_jepa_world_models.exercises.part1_jepa_world_model_controls import solutions

## Exercise 1 - Paired Cosine

<details>
<summary>Expected output</summary>

```text
All tests in `test_paired_cosine_toy_oracle` passed!
```

</details>

In [2]:
paired_cosine = solutions.paired_cosine

tests.test_paired_cosine_toy_oracle(paired_cosine)
tests.test_paired_cosine_rejects_shape_mismatch(paired_cosine)

All tests in `test_paired_cosine_toy_oracle` passed!
All tests in `test_paired_cosine_rejects_shape_mismatch` passed!


## Exercise 2 - Target Prediction and Collapse

The target predictor passes on exact toy embeddings; collapsed and wrong-scale controls fail.

In [3]:
prediction = solutions.jepa_prediction_smoke_test()
collapse = solutions.collapse_diagnostics_smoke_test()
utils.print_report("JEPA target prediction", prediction)
print("collapsed_control_rejected:", collapse["collapsed_control_rejected"])

tests.test_jepa_prediction_smoke_test(solutions.jepa_prediction_smoke_test)
tests.test_jepa_prediction_report_rejects_collapse_and_bad_mse()
tests.test_collapse_diagnostics_smoke_test(solutions.collapse_diagnostics_smoke_test)
tests.test_collapse_diagnostics_rejects_identical_features()

JEPA target prediction
  mean_cosine     : 1.0
  mse             : 0.0
  predicts_target : True
collapsed_control_rejected: True
All tests in `test_jepa_prediction_smoke_test` passed!
All tests in `test_jepa_prediction_report_rejects_collapse_and_bad_mse` passed!
All tests in `test_collapse_diagnostics_smoke_test` passed!
All tests in `test_collapse_diagnostics_rejects_identical_features` passed!


## Exercise 3 - Held-Out State Probes

<details>
<summary>Interpretation</summary>

The same logits must fail after label shuffling. Otherwise the probe score is not evidence about world state.

</details>

In [4]:
state_probe = solutions.state_probe_smoke_test()
state_control = solutions.state_probe_control_smoke_test()
utils.print_report("State probe", state_probe)
print("shuffled_control_rejected:", state_control["shuffled_control_rejected"])
print("accuracy_margin:", state_control["accuracy_margin"])

tests.test_state_probe_smoke_test(solutions.state_probe_smoke_test)
tests.test_state_probe_control_smoke_test(solutions.state_probe_control_smoke_test)
tests.test_state_probe_report_rejects_shuffled_labels()

State probe
  accuracy       : 1.0
  predicts_state : True
shuffled_control_rejected: True
accuracy_margin: 1.0
All tests in `test_state_probe_smoke_test` passed!
All tests in `test_state_probe_control_smoke_test` passed!
All tests in `test_state_probe_report_rejects_shuffled_labels` passed!


## Exercise 4 - Transition and Rollout Controls

A transition check compares `state + action_delta` to the next latent. A rollout check requires action-conditioned prediction to beat copy and shuffled-action baselines.

In [5]:
transition = solutions.transition_smoke_test()
rollout = solutions.rollout_control_smoke_test()
utils.print_report("Transition", transition)
print("rollout_passes:", rollout["rollout"]["rollout_passes"])
print("copy_and_shuffled_controls_rejected:", rollout["copy_and_shuffled_controls_rejected"])

tests.test_transition_smoke_test(solutions.transition_smoke_test)
tests.test_transition_report_rejects_missing_action_delta()
tests.test_rollout_control_smoke_test(solutions.rollout_control_smoke_test)
tests.test_latent_rollout_report_rejects_copy_and_shuffled_controls()

Transition
  mean_cosine           : 1.0
  transition_consistent : True
rollout_passes: True
copy_and_shuffled_controls_rejected: True
All tests in `test_transition_smoke_test` passed!
All tests in `test_transition_report_rejects_missing_action_delta` passed!
All tests in `test_rollout_control_smoke_test` passed!
All tests in `test_latent_rollout_report_rejects_copy_and_shuffled_controls` passed!


## Exercise 5 - Object Permanence Controls

Occluded-object evidence must remain above absent-object evidence, and different-object similarity must fail.

In [6]:
permanence = solutions.object_permanence_smoke_test()
permanence_control = solutions.object_permanence_control_smoke_test()
utils.print_report("Object permanence", permanence)
print("absent_like_rejected:", permanence_control["absent_like_rejected"])
print("different_object_rejected:", permanence_control["different_object_rejected"])

tests.test_object_permanence_smoke_test(solutions.object_permanence_smoke_test)
tests.test_object_permanence_control_smoke_test(solutions.object_permanence_control_smoke_test)
tests.test_object_permanence_report_rejects_absent_and_different_object_controls()

Object permanence
  visible_mean              : 0.8500000238418579
  occluded_mean             : 0.7250000238418579
  absent_mean               : 0.15000000596046448
  occluded_absent_gap       : 0.5750000178813934
  preserves_occluded_object : True
absent_like_rejected: True
different_object_rejected: True
All tests in `test_object_permanence_smoke_test` passed!
All tests in `test_object_permanence_control_smoke_test` passed!
All tests in `test_object_permanence_report_rejects_absent_and_different_object_controls` passed!


## Exercise 6 - Notebook Contract

The solved contract collects every toy report into one JSON-serializable dictionary.

In [7]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["jepa_prediction"]["predicts_target"]
assert contract["collapse"]["collapsed_control_rejected"]
assert contract["state_probe_control"]["shuffled_control_rejected"]
assert contract["rollout_control"]["copy_and_shuffled_controls_rejected"]
assert contract["object_permanence_control"]["different_object_rejected"]
tests.test_notebook_contract(solutions.run_smoke_test)
contract

All tests in `test_notebook_contract` passed!


{'jepa_prediction': {'mean_cosine': 1.0, 'mse': 0.0, 'predicts_target': True},
 'collapse': {'structured': {'finite_features': True,
   'feature_std': 0.4815434217453003,
   'effective_rank': 3.2464075088500977,
   'non_collapsed': True},
  'collapsed': {'finite_features': True,
   'feature_std': 0.0,
   'effective_rank': 1.0,
   'non_collapsed': False},
  'collapsed_control_rejected': True},
 'state_probe': {'accuracy': 1.0, 'predicts_state': True},
 'state_probe_control': {'probe': {'accuracy': 1.0, 'predicts_state': True},
  'shuffled': {'accuracy': 0.0, 'predicts_state': False},
  'shuffled_control_rejected': True,
  'accuracy_margin': 1.0},
 'transition': {'mean_cosine': 1.0, 'transition_consistent': True},
 'rollout_control': {'rollout': {'rollout_loss': 0.1,
   'copy_baseline_loss': 1.0,
   'shuffled_action_loss': 0.9,
   'beats_copy_baseline': True,
   'shuffled_action_fails': True,
   'rollout_passes': True},
  'failed_control': {'rollout_loss': 0.75,
   'copy_baseline_loss': 

## Signature Result

| Check | Observed | Required |
|---|---:|---:|
| Same-object margin | `0.0488` | `>= 0.030` |
| Occluded-vs-absent gap | `0.2230` | `>= 0.200` |
| Masked prediction loss reduction | `0.9992` | `>= 0.500` |
| Probe margin over shuffled | `0.5400` | `>= 0.200` |
| Object-token patch gap | `0.99997` | `>= 0.400` |

## Limitations

### What this does not show

This is not a real-video object-permanence benchmark, not V-JEPA fine-tuning, and not a replication of I-JEPA, VL-JEPA, Othello-GPT, maze, Sudoku, or RL world models.

## Bonus - Anomaly Hunting

- Move the generated object to unseen x positions and check the probe margin.
- Randomize only background tokens and confirm the patching effect stays small.
- Replace the absent-object video with a distractor object and narrow the claim if the gap vanishes.

In [8]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert report["accepted"]
assert gpu["cuda_available"]
assert gpu["vjepa2_preflight_passed"]
assert gpu["vjepa2_world_model_controls_passed"]
assert gpu["peak_vram_gb"] < 24.0
print("vjepa2_feature_shape:", gpu["vjepa2_feature_shape"])
print("vjepa2_world_feature_shape:", gpu["vjepa2_world_feature_shape"])
print("masked_prediction_loss_reduction:", round(gpu["masked_prediction_loss_reduction"], 3))
print("state_probe_margin_over_random:", round(gpu["state_probe_margin_over_random"], 3))
print("causal_latent_patch_random_gap:", round(gpu["causal_latent_patch_random_gap"], 3))
print("peak_vram_gb:", round(gpu["peak_vram_gb"], 3))

tests.test_committed_verification_report_vjepa_world_controls()
tests.test_exercise_notebook_declares_full_verification_contract()

vjepa2_feature_shape: [5, 144, 1024]
vjepa2_world_feature_shape: [200, 1024]
masked_prediction_loss_reduction: 0.999
state_probe_margin_over_random: 0.54
causal_latent_patch_random_gap: 1.0
peak_vram_gb: 0.74
All tests in `test_committed_verification_report_vjepa_world_controls` passed!
All tests in `test_exercise_notebook_declares_full_verification_contract` passed!
